In [ ]:
# Import dependencies
from pathlib import Path

import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler


# Define feature engineering function
def create_features(df):
    # Create copy
    df = df.copy()

    # =========================================================
    # Tire related features
    # =========================================================

    # Tire age efficiency
    if (
        "TireAge" in df.columns
        and "LapNumber" in df.columns
    ):
        df["TireAgeRatio"] = (
            df["TireAge"]
            / (
                df["LapNumber"] + 1
            )
        )

    # Tire degradation pressure
    if (
        "TireAge" in df.columns
        and "TrackTemp" in df.columns
    ):
        df["TireTempInteraction"] = (
            df["TireAge"]
            * df["TrackTemp"]
        )

    # =========================================================
    # Position and race context
    # =========================================================

    # Relative race progression
    if (
        "LapNumber" in df.columns
        and "Position" in df.columns
    ):
        df["RacePositionPressure"] = (
            df["LapNumber"]
            * df["Position"]
        )

    # Pit strategy pressure
    if (
        "GapToLeader" in df.columns
        and "Position" in df.columns
    ):
        df["LeaderPressure"] = (
            df["GapToLeader"]
            * df["Position"]
        )

    # =========================================================
    # Speed related interactions
    # =========================================================

    # Sector consistency
    sector_cols = [
        col
        for col in [
            "Sector1Time",
            "Sector2Time",
            "Sector3Time",
        ]
        if col in df.columns
    ]

    # Compute sector stats
    if len(sector_cols) > 0:
        # Mean sector time
        df["SectorMean"] = (
            df[sector_cols]
            .mean(axis=1)
        )

        # Sector standard deviation
        df["SectorStd"] = (
            df[sector_cols]
            .std(axis=1)
        )

        # Sector range
        df["SectorRange"] = (
            df[sector_cols]
            .max(axis=1)
            - df[sector_cols]
            .min(axis=1)
        )

    # =========================================================
    # Weather and temperature
    # =========================================================

    # Air / track delta
    if (
        "AirTemp" in df.columns
        and "TrackTemp" in df.columns
    ):
        df["TrackAirDelta"] = (
            df["TrackTemp"]
            - df["AirTemp"]
        )

    # Humidity interaction
    if (
        "Humidity" in df.columns
        and "TrackTemp" in df.columns
    ):
        df["HumidityTrackInteraction"] = (
            df["Humidity"]
            * df["TrackTemp"]
        )

    # =========================================================
    # Rolling statistical features
    # =========================================================

    # Numerical columns
    numerical_cols = (
        df.select_dtypes(
            include=np.number
        )
        .columns.tolist()
    )

    # Remove id if present
    numerical_cols = [
        col
        for col in numerical_cols
        if col != "id"
    ]

    # Compute row statistics
    if len(numerical_cols) > 0:
        # Row mean
        df["RowMean"] = (
            df[numerical_cols]
            .mean(axis=1)
        )

        # Row std
        df["RowStd"] = (
            df[numerical_cols]
            .std(axis=1)
        )

        # Row min
        df["RowMin"] = (
            df[numerical_cols]
            .min(axis=1)
        )

        # Row max
        df["RowMax"] = (
            df[numerical_cols]
            .max(axis=1)
        )

        # Row range
        df["RowRange"] = (
            df["RowMax"]
            - df["RowMin"]
        )

    # Return dataframe
    return df


# Define preprocessing function
def preprocess_data(
    train_df,
    test_df,
    target_col,
):
    # Apply feature engineering
    train_df = create_features(
        train_df
    )

    # Apply feature engineering
    test_df = create_features(
        test_df
    )

    # Split features
    X = train_df.drop(
        columns=[target_col]
    )

    # Extract target
    y = train_df[target_col]

    # Save test ids
    test_ids = test_df["id"]

    # Remove id column
    X = X.drop(columns=["id"])

    # Remove id from test
    X_test = test_df.drop(
        columns=["id"]
    )

    # Detect categorical columns
    categorical_cols = (
        X.select_dtypes(
            include=[
                "object",
                "category",
            ]
        )
        .columns.tolist()
    )

    # Detect numerical columns
    numerical_cols = (
        X.select_dtypes(
            exclude=[
                "object",
                "category",
            ]
        )
        .columns.tolist()
    )

    # Define numerical transformer
    numerical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="median"
                ),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    # Define categorical transformer
    categorical_transformer = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="most_frequent"
                ),
            ),
            (
                "encoder",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False,
                ),
            ),
        ]
    )

    # Define preprocessor
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                numerical_transformer,
                numerical_cols,
            ),
            (
                "cat",
                categorical_transformer,
                categorical_cols,
            ),
        ]
    )

    # Fit and transform train
    X_processed = (
        preprocessor.fit_transform(X)
    )

    # Transform test
    X_test_processed = (
        preprocessor.transform(X_test)
    )

    # Print feature count
    print(
        f"Processed features: "
        f"{X_processed.shape[1]}"
    )

    # Return processed data
    return (
        X_processed,
        y.values,
        X_test_processed,
        test_ids,
    )


# Define training function
def train_model(
    X_train,
    y_train,
    X_valid,
    y_valid,
):
    # Define model
    model = XGBRegressor(
        n_estimators=5000,
        learning_rate=0.005,
        max_depth=9,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.25,
        reg_lambda=8.0,
        gamma=0.1,

        # Objective
        objective="reg:absoluteerror",

        # GPU acceleration
        tree_method="hist",
        device="cuda",

        # Misc
        random_state=42,
        n_jobs=-1,
    )

    # Train model
    model.fit(
        X_train,
        y_train,
        eval_set=[
            (
                X_valid,
                y_valid,
            )
        ],
        verbose=100,
    )

    # Predict validation
    valid_predictions = model.predict(
        X_valid
    )

    # Compute validation MAE
    valid_mae = mean_absolute_error(
        y_valid,
        valid_predictions,
    )

    # Print score
    print(
        f"Validation MAE: "
        f"{valid_mae:.5f}"
    )

    # Return model
    return model


# Define prediction function
def predict_model(
    model,
    X_test,
):
    # Predict
    predictions = model.predict(
        X_test
    )

    # Return predictions
    return predictions


# Define blending function
def blend_predictions(
    test_ids,
    prediction_dict,
    output_path,
    target_col,
):
    # Initialize weighted sum
    weighted_sum = np.zeros(
        len(test_ids)
    )

    # Initialize total weight
    total_weight = 0.0

    # Iterate predictions
    for name, config in (
        prediction_dict.items()
    ):
        # Extract predictions
        predictions = config[
            "predictions"
        ]

        # Extract weight
        weight = config[
            "weight"
        ]

        # Add weighted predictions
        weighted_sum += (
            predictions * weight
        )

        # Accumulate weight
        total_weight += weight

        # Print info
        print(
            f"Blending {name} | "
            f"Weight: {weight}"
        )

    # Compute final predictions
    final_predictions = (
        weighted_sum / total_weight
    )

    # Create submission
    submission = pd.DataFrame(
        {
            "id": test_ids,
            target_col: final_predictions,
        }
    )

    # Save submission
    submission.to_csv(
        output_path,
        index=False,
    )

    # Print confirmation
    print(
        f"✅ Submission saved to "
        f"{output_path}"
    )


# Define the main function
def main():
    # Define target
    TARGET = "PitNextLap"

    # Define competition path
    COMP_PATH = Path(
        "/kaggle/input/"
        "competitions/"
        "playground-series-s6e5"
    )

    # Define blend dataset path
    BLEND_PATH = Path(
        "/kaggle/input/"
        "datasets/"
        "anthonytherrien/"
        "predicting-f1-pit-stops-vault"
    )

    # Load train dataset
    train_df = pd.read_csv(
        COMP_PATH / "train.csv"
    )

    # Load test dataset
    test_df = pd.read_csv(
        COMP_PATH / "test.csv"
    )

    # Preprocess data
    (
        X,
        y,
        X_test,
        test_ids,
    ) = preprocess_data(
        train_df,
        test_df,
        TARGET,
    )

    # Split validation
    (
        X_train,
        X_valid,
        y_train,
        y_valid,
    ) = train_test_split(
        X,
        y,
        test_size=0.1,
        random_state=42,
    )

    # Train model
    model = train_model(
        X_train=X_train,
        y_train=y_train,
        X_valid=X_valid,
        y_valid=y_valid,
    )

    # Predict XGBoost
    xgb_predictions = predict_model(
        model=model,
        X_test=X_test,
    )

    # Define prediction dictionary
    prediction_dict = {
        "sub1": {
            "predictions": pd.read_csv(
                BLEND_PATH
                / "submission.csv"
            )[TARGET].values,
            "weight": 2.9,
        },
        "sub2": {
            "predictions": pd.read_csv(
                BLEND_PATH
                / "submission (1).csv"
            )[TARGET].values,
            "weight": 0.1,
        },
        "xgb": {
            "predictions": xgb_predictions,
            "weight": 0.000001,
        },
    }

    # Blend predictions
    blend_predictions(
        test_ids=test_ids,
        prediction_dict=prediction_dict,
        output_path="submission.csv",
        target_col=TARGET,
    )


# Call the main function
if __name__ == "__main__":
    main()